## 1) Move your body (aka direct geometry)

在这一系列的首个练习中，我们将开始使用 Pinocchio 库。我们将在模拟器中加载一个简单的机械臂模型，即 Universal Robot 5（简称 UR5）。该模型将用于通过简单方法完成定位任务。这套练习重点在于课程的第一部分，即直接几何学。  
Pinocchio 是一个 C++ 库，附带 Python 封装，允许我们通过 Python 终端进行控制。让我们看看它是如何工作的。

本教程需要 Pinocchio、Gepetto GUI 以及 ur5 机器人的描述文件。



最简单的方法是添加 robotpkg apt 仓库并运行：



In [ ]:
!sudo apt install robotpkg-pinocchio robotpkg-ur5-description robotpkg-py310-qt5-gepetto-viewer

接下来，我们将使用 numpy 的 Matrix 类来表示矩阵和向量。在 numpy 中，向量就是只有一列的矩阵。请看下面的例子。

In [1]:
import numpy as np
A = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])  # Define a 2x4 matrix
A = np.array([[1, 2, 3, 4], [5, 6, 7, 8]])  # 定义一个 2x4 矩阵
b = np.zeros([4, 1])  # Define a 4 vector (ie a 4x1 matrix) initialized with 0
b = np.zeros([4, 1])  # 定义一个 4 维向量（即一个 4x1 矩阵），初始化为 0
c = A @ b             # Obtain c by multiplying A by b.
c = A @ b             # 通过将 A 乘以 b 得到 c。

许多实用函数都封装在 pinocchio 的 utils 中。



In [2]:
import pinocchio as pin
from pinocchio.utils import *
import numpy as np
eye(6)                      # Return a 6x6 identity matrix
eye(6)                      # 返回一个 6x6 的单位矩阵
zero(6)                     # Return a zero 6x1 vector
zero(6)                     # 返回一个 6x1 的零向量
zero([6, 4])                # Return az zero 6x4 matrix
zero([6, 4])                # 返回一个 6x4 的零矩阵
rand(6)                     # Random 6x1 vector
rand(6)                     # 随机 6x1 向量
isapprox(zero(6), rand(6))  # Test epsilon equality
isapprox(zero(6), rand(6))  # 测试 epsilon 相等性
mprint(rand([6, 6]))        # Matlab-style print
mprint(rand([6, 6]))        # Matlab 风格打印
pin.skew(rand(3))               # Skew "cross-product" 3x3 matrix from a 3x1 vector
pin.skew(rand(3))               # 从 3x1 向量生成斜对称"叉积"3x3 矩阵
np.cross(rand(3), rand(3))     # Cross product of R^3
np.cross(rand(3), rand(3))     # R^3 空间的叉积
rotate('x', 0.4)            # Build a rotation matrix of 0.4rad around X.
rotate('x', 0.4)            # 构建一个绕 X 轴旋转 0.4 弧度的旋转矩阵。

ans  = 

Columns 0 through 5

   0.71916    0.86282    0.75849    0.42029    0.51111    0.06090   
   0.71750    0.25999    0.57227    0.86591    0.26946    0.48023   
   0.00270    0.29884    0.14404    0.28844    0.99495    0.90191   
   0.81567    0.93719    0.75440    0.61787    0.42713    0.01418   
   0.79133    0.04218    0.43300    0.08157    0.09507    0.06282   
   0.78949    0.07368    0.85894    0.12350    0.46281    0.48838   

ans  = 

Columns 0 through 5

   0.42897    0.44894    0.28778    0.76347    0.30188    0.46504   
   0.37505    0.55291    0.06984    0.47499    0.14650    0.73096   
   0.01829    0.23817    0.88677    0.22272    0.66520    0.50889   
   0.97403    0.43202    0.37642    0.14921    0.56063    0.94356   
   0.84323    0.18797    0.19847    0.56355    0.69674    0.43869   
   0.53141    0.92976    0.02533    0.40644    0.87671    0.42192   



array([[ 1.        ,  0.        ,  0.        ],
       [ 0.        ,  0.92106099, -0.38941834],
       [ 0.        ,  0.38941834,  0.92106099]])

定义了特定的类来表示 \(SE(3)\)、\(se(3)\) 和 \(se(3)^*\) 的对象。刚性位移，即 \(SE(3)\) 的元素，由类 SE3 表示。

In [3]:
import pinocchio as pin
R = eye(3)
p = zero(3)
M0 = pin.SE3(R, p)
M = pin.SE3.Random()
M.translation = p
M.rotation = R

空间速度，作为 \(se(3) = M^6\) 的元素，由类 Motion 表示。



In [5]:
v = zero(3)
w = zero(3)
nu0 = pin.Motion(v, w)
nu = pin.Motion.Random()
nu.linear = v
nu.angular = w

空间力，即 \(se(3)^* = F^6\) 的元素，由类 Force 表示。



In [6]:
f = zero(3)
tau = zero(3)
phi0 = pin.Force(f, tau)
phi = pin.Force.Random()
phi.linear = f
phi.angular = tau


### 1.1) 创建并显示机器人

#### Robot kinematic tree  机器人运动学树

运动学树由两个 C++对象表示：Model（包含模型常量：长度、质量、名称等）和 Data（包含模型算法使用的工作内存）。这两个 C++对象都包含在一个唯一的 Python 类中。load_robot_description 可以顺利地将常见机器人模型加载到 pinocchio 中。首先使用以下命令安装 example-robot-data 包：'sudo apt install robotpkg-example-robot-data'

In [7]:
from robot_descriptions.loaders.pinocchio import load_robot_description

robot = load_robot_description("ur5_description")
model = robot.model
collision_model = robot.collision_model
visual_model = robot.visual_model

Cloning https://github.com/Gepetto/example-robot-data.git...


100%|██████████| 2413.0/2413.0 [01:13<00:00, 32.78it/s]  


RobotWrapper 类的代码也可用于加载本地 URDF 文件，相关代码位于 /opt/openrobots/lib/python3.10/site-packages/pinocchio/robot_wrapper.py 。欢迎随时查阅该实现，并参考类函数的设计思路获取灵感。



UR5 是由丹麦公司 Universal Robot 开发的一款固定式机器人，配备一条 6 自由度机械臂。其所有 6 个关节均为旋转关节。该机器人的构型位于 R^6 空间，且不受任何约束限制。UR5 的模型通过 URDF 文件进行描述，其中机器人各部位的视觉外观采用 Collada 格式的".dae"文件以网格形式（即多边形集合）呈现。URDF 和 DAE 文件均可在‘robotpkg-example-robot-data’软件包中获取。

## Exploring the model  探索模型


机器人模型可在 robot.model 中获取。它包含所有机器人关节的名称 names 、运动学树 parents （即父节点图，0 为根节点且无父节点）、当前关节在父坐标系中的位置 jointPosition 、所有刚体的质量、惯性和重心位置（浓缩为 6x6 空间惯性矩阵） inertias ，以及关联世界的重力 gravity 。所有这些功能均有文档记录，可在相应的类字典中找到。



In [8]:
for name, function in robot.model.__class__.__dict__.items():
    print(' **** %s: %s' % (name, function.__doc__))

 **** __module__: str(object='') -> str
str(bytes_or_buffer[, encoding[, errors]]) -> str

Create a new string object from the given object. If encoding or
errors is specified, then the object must expose a data buffer
that will be decoded using the given encoding and error handler.
Otherwise, returns the result of object.__str__() (if defined)
or repr(object).
encoding defaults to sys.getdefaultencoding().
errors defaults to 'strict'.
 **** __doc__: str(object='') -> str
str(bytes_or_buffer[, encoding[, errors]]) -> str

Create a new string object from the given object. If encoding or
errors is specified, then the object must expose a data buffer
that will be decoded using the given encoding and error handler.
Otherwise, returns the result of object.__str__() (if defined)
or repr(object).
encoding defaults to sys.getdefaultencoding().
errors defaults to 'strict'.
 **** __reduce__: None
 **** __init__: 
__init__( (object)self) -> None :
    Default constructor. Constructs an empty mode

同样，机器人数据可在 robot.data 中获取。所有由经典刚体动力学算法分配的变量都存储在 robot.data 中，并通过 Python 封装层提供访问。与模型对象类似，这些函数均有文档说明，可通过类字典访问。后续最实用的将是存储在 robot.data.oMi 中的各关节输出关联坐标系位姿信息。



例如，机器人末端执行器对应于最后一个关节的输出，称为 wrist_1_joint 。关节列表中该关节的 ID 可以从其名称中恢复，然后用于访问其位姿：



In [15]:
# Get index of end effector
# 获取末端执行器索引

idx = robot.index('wrist_3_joint')

# Compute and get the placement of joint number idx
# 计算并获取编号为 idx 的关节的位置
q = pin.randomConfiguration(model)
print(f"q: {q.T}")
q1 = np.asarray([1,1,1,1,1,1])
placement = robot.placement(q1, idx)
# Be carreful, Python always returns references to values.
# 注意，Python 总是返回值的引用。
# You can often .copy() the object to avoid side effects
# 通常可以通过 .copy() 方法复制对象来避免副作用
# Only get the placement
# 仅获取位姿
placement = robot.data.oMi[idx].copy()

q: [-3.86775488  2.05117006  2.45190375 -1.89886738 -5.47678471 -6.03156825]


最后，一些在模型和数据中反复使用的数据已被封装到一些 Python 快捷方式的函数中，这些函数在 RomeoWrapper 中也可用：
- 机器人配置的大小由 nq 给出。
- 其切空间（速度）的维度为 nv 。
- 树中关节的索引可以通过其名称使用 index 函数访问（见上文）。
- 经典算法也已绑定：质心、质心雅可比矩阵、质量、偏置、关节重力、各关节的位置与速度。

In [18]:
q = zero(robot.nq)
v = rand(robot.nv)
robot.com(q)  # Compute the robot center of mass.
robot.placement(q, 3)  # Compute the placement of joint 3

SE3(array([[ 4.89663865e-12,  0.00000000e+00,  1.00000000e+00,  4.25000000e-01],[ 0.00000000e+00,  1.00000000e+00,  0.00000000e+00,  1.61500000e-02],[-1.00000000e+00,  0.00000000e+00,  4.89663865e-12,  8.91590000e-02],[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00]]))

## 显示机器人


要显示机器人，我们需要一个名为 Gepetto Viewer 的外部程序。如果您已完成上一页的安装，可以启动此程序，在空白工作区中打开一个新终端。

这将启动一个等待指令的服务器。现在我们将创建一个客户端，请求服务器执行一些操作（例如创建窗口或显示我们的机器人）

在 Python 终端中，您现在可以在查看器中加载机器人的视觉模型：



In [38]:
from pinocchio.visualize import GepettoVisualizer, MeshcatVisualizer
VISUALIZER = GepettoVisualizer
robot.setVisualizer(VISUALIZER())
robot.initViewer(loadModel=True)
robot.loadViewerModel("pinocchio")
q0 = robot.q0
robot.display(q0)


In [36]:
visualObj = robot.visual_model.geometryObjects[4]  # 3D object representing the robot forarm
visualName = visualObj.name                        # Name associated to this object
visualRef = robot.getViewerNodeName(visualObj, pin.GeometryType.VISUAL)    # Viewer reference (string) representing this object


移动一个物体

In [37]:
q1 = (1, 1, 1, 1, 0, 0, 0)  # x, y, z, 四元数

robot.viewer.gui.applyConfiguration(visualRef, q1)
robot.viewer.gui.refresh()  # Refresh the window.

In [39]:
rgbt = [1.0, 0.2, 0.2, 1.0]  # 红色、绿色、蓝色、透明度
robot.viewer.gui.addSphere("world/sphere", .1, rgbt)  # .1 是半径


True

### 1.2) 简单的拾取与放置


目标：在给定配置或沿给定轨迹显示机器人

假设我们有一个位于位置 [.5, .1, .2] 的目标，我们希望机器人能够抓取它。



In [40]:
robot.viewer.gui.applyConfiguration("world/sphere", (.5, .1, .2, 1.,0.,0.,0. ))
robot.viewer.gui.refresh()  # Refresh the window.

首先在此位置显示一个小球体以进行可视化。



然后通过任意方式决定机器人的配置，使末端执行器接触到球体。



在你构建的参考位置下，末端执行器的位姿可通过 robot.placement(q, 6) 获取。这里仅选取了位姿的平移部分，旋转部分保持自由。



假设现在物体是矩形而非球体。在参考位置拾取物体时需遵循强制旋转约束，使末端执行器与矩形的一个面对齐。



在配置空间中选择任意你想要的轨迹，从上一个练习中建立的参考位置开始（可以是正弦-余弦波、多项式、样条曲线、直线等）。



使用一个 for 循环，沿着该轨迹在采样位置显示机器人。可以使用 time 模块中的 sleep 函数（通过 from time import sleep 导入）来减慢循环速度。



在循环的每个瞬间，重新计算球的位置并显示它，使其始终“粘附”在机器人末端执行器上。



In [50]:
import math
import time
q = zero(robot.nq)
v = rand(robot.nv)
robot.com(q)  # Compute the robot center of mass.
robot.placement(q, 3)  # Compute the placement of joint 3

count = 0
while True:
    for i in range(robot.nq):
        q[i] = math.sin(count * 0.1+i * 0.1)
        # print(q.T)
    robot.display(q)
    idx = robot.index('wrist_3_joint')
    placement = robot.placement(q, idx)
    sphere_cfg = pin.SE3ToXYZQUAT(placement)  # [x, y, z, qx, qy, qz, qw]
    robot.viewer.gui.applyConfiguration("world/sphere", tuple(sphere_cfg))
    robot.viewer.gui.refresh()  # Refresh the window.
    count += 1
    time.sleep(0.1)

KeyboardInterrupt: 